In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# %% 셀 0: 데이터 로드
import os, json, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10
MAX_ACTIVE = 150
MAX_TEXT_LEN = 200
SCALAR_DIM = 16
SEG_SCALAR_DIM = 4

text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩: {len(text2emb):,}개 dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None
    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])
        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f
    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)
    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))
    video_name = data.get("video", "")
    file_id = os.path.basename(path)[:-5]
    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))
        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })
    stt_path = os.path.join(STT_DIR, channel, file_id + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass
    return {
        "channel": channel, "video_name": video_name, "file_id": file_id,
        "instances": inst_list, "stt_segments": stt_segments, "duration": duration,
    }


json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

samples = []
channel_set = set()
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

rng = random.Random(SEED)
by_channel = {}
for s in samples:
    by_channel.setdefault(s["channel"], []).append(s)

train_samples = []
eval_samples = []
for ch, ch_samples in by_channel.items():
    ch_samples.sort(key=lambda s: s["file_id"])
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

train_samples = [s for s in train_samples if len(s["instances"]) > 0]
eval_samples  = [s for s in eval_samples  if len(s["instances"]) > 0]

print(f"✅ 채널: {len(channels)}개  train: {len(train_samples):,}  eval: {len(eval_samples):,}")


✅ 임베딩: 2,167,019개 dim=1024


로드: 100%|██████████| 66400/66400 [00:14<00:00, 4517.39it/s]


✅ 채널: 664개  train: 56,892  eval: 2,984


In [2]:
# %% 셀 1: 채널 서브샘플링
rng_ch = random.Random(42)
sampled_channels = rng_ch.sample(channels, 67)
sampled_set = set(sampled_channels)

train_samples = [s for s in train_samples if s["channel"] in sampled_set]
eval_samples  = [s for s in eval_samples  if s["channel"] in sampled_set]
channels = sampled_channels
channel2id = {ch: i for i, ch in enumerate(channels)}

print(f"🔬 {len(channels)}개 채널  train: {len(train_samples):,}  eval: {len(eval_samples):,}")


🔬 67개 채널  train: 5,574  eval: 288


In [3]:
# %% 셀 1.5: 채널별 레이아웃 통계 (train set에서만)
ch_data = {ch: {"cx": [], "cy": [], "w": [], "h": []} for ch in channels}
for s in train_samples:
    ch = s["channel"]
    for inst in s["instances"]:
        ch_data[ch]["cx"].append(inst["x"] / GRID_W)
        ch_data[ch]["cy"].append(inst["y"] / GRID_H)
        ch_data[ch]["w"].append(inst["w"] / GRID_W)
        ch_data[ch]["h"].append(inst["h"] / GRID_H)

ch_stats = {}
for ch in channels:
    d = ch_data[ch]
    cxs = np.array(d["cx"]); cys = np.array(d["cy"])
    ws  = np.array(d["w"]);  hs  = np.array(d["h"])
    ch_stats[ch] = np.array([
        cxs.mean(), cys.mean(), ws.mean(), hs.mean(),
        cxs.std(),  cys.std(),  ws.std(),  hs.std(),
    ], dtype=np.float32)

print(f"📊 채널 통계: {len(ch_stats)}개")


📊 채널 통계: 67개


In [4]:
# %% 셀 2: SegmentDataset
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 512
NUM_WORKERS_DL = 8
MAX_SEG_LOG = math.log(MAX_ACTIVE * 100)


def make_segments(s):
    insts = s["instances"]
    duration = max(s["duration"], 0.1)
    bounds = {0.0, duration}
    for inst in insts:
        bounds.add(inst["start"])
        bounds.add(inst["end"])
    bounds = sorted(b for b in bounds if 0 <= b <= duration)
    segments = []
    for t0, t1 in zip(bounds, bounds[1:]):
        if t1 - t0 <= 1e-4:
            continue
        mid = (t0 + t1) / 2
        active_idx = [j for j, inst in enumerate(insts)
                      if inst["start"] <= mid < inst["end"]]
        if not active_idx:
            continue
        if len(active_idx) > MAX_ACTIVE:
            active_idx = sorted(active_idx,
                                key=lambda j: -insts[j]["text_len"])[:MAX_ACTIVE]
        segments.append({
            "t0": t0, "t1": t1, "mid": mid,
            "active_idx": active_idx,
            "seg_len_frames": max(1.0, (t1 - t0) * FPS),
        })
    return segments


def build_segment_index(video_samples):
    entries = []
    for s in tqdm(video_samples, desc="segment 분해"):
        for seg in make_segments(s):
            entries.append((s, seg))
    return entries


train_entries = build_segment_index(train_samples)
eval_entries  = build_segment_index(eval_samples)
print(f"📦 train: {len(train_entries):,}  eval: {len(eval_entries):,} segments")


class SegmentDataset(Dataset):
    def __init__(self, entries, channel2id, text2emb, ch_stats):
        self.entries = entries
        self.channel2id = channel2id
        self.text2emb = text2emb
        self.ch_stats = ch_stats

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        s, seg = self.entries[idx]
        insts = s["instances"]
        active_idx = seg["active_idx"]
        N = len(active_idx)
        duration = max(s["duration"], 0.1)
        mid = seg["mid"]
        ch_stat = self.ch_stats[s["channel"]]

        channel_emb = F.normalize(self.text2emb.get(s["channel"],    ZERO_EMB), dim=-1)
        video_emb   = F.normalize(self.text2emb.get(s["video_name"], ZERO_EMB), dim=-1)

        stt_emb = ZERO_EMB; has_stt = 0.0
        for sg in s["stt_segments"]:
            if sg["start"] <= mid < sg["end"]:
                stt_emb = F.normalize(self.text2emb.get(sg["text"], ZERO_EMB), dim=-1)
                has_stt = 1.0
                break

        active_insts = [insts[j] for j in active_idx]
        inst_embs = torch.stack([
            F.normalize(self.text2emb.get(inst["text"], ZERO_EMB), dim=-1)
            for inst in active_insts
        ])
        diff_ch  = inst_embs - channel_emb.unsqueeze(0)
        diff_vid = inst_embs - video_emb.unsqueeze(0)
        diff_stt = inst_embs - stt_emb.unsqueeze(0)

        ch_sims  = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
        vid_sims = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0),   dim=-1).numpy()
        stt_sims = F.cosine_similarity(inst_embs, stt_emb.unsqueeze(0),     dim=-1).numpy()

        inst_scalars = np.zeros((N, SCALAR_DIM), dtype=np.float32)
        inst_scalars[:, 0] = [inst["text_len"] / MAX_TEXT_LEN for inst in active_insts]
        inst_scalars[:, 1] = ch_sims
        inst_scalars[:, 2] = vid_sims
        inst_scalars[:, 3] = stt_sims
        inst_scalars[:, 4] = has_stt
        inst_scalars[:, 5] = [inst["start"] / duration for inst in active_insts]
        inst_scalars[:, 6] = [inst["end"]   / duration for inst in active_insts]
        inst_scalars[:, 7] = [(inst["end"] - inst["start"]) / duration for inst in active_insts]
        inst_scalars[:, 8:16] = ch_stat[None, :]

        seg_scalar = np.array([
            mid / duration,
            (seg["t1"] - seg["t0"]) / duration,
            min(N / MAX_ACTIVE, 1.0),
            np.log1p(seg["seg_len_frames"]) / MAX_SEG_LOG,
        ], dtype=np.float32)

        gt_bbox = np.zeros((N, 4), dtype=np.float32)
        for i, inst in enumerate(active_insts):
            cx, cy, w, h = inst["x"], inst["y"], inst["w"], inst["h"]
            x0 = max(0, cx - w // 2); y0 = max(0, cy - h // 2)
            x1 = min(GRID_W, x0 + w); y1 = min(GRID_H, y0 + h)
            gt_bbox[i] = [x0, y0, x1, y1]

        return {
            "channel_id":   self.channel2id[s["channel"]],
            "inst_embs":    inst_embs,
            "diff_ch":      diff_ch,
            "diff_vid":     diff_vid,
            "diff_stt":     diff_stt,
            "inst_scalars": torch.from_numpy(inst_scalars),
            "seg_scalar":   torch.from_numpy(seg_scalar),
            "channel_emb":  channel_emb,
            "video_emb":    video_emb,
            "stt_emb":      stt_emb,
            "gt_bbox":      torch.from_numpy(gt_bbox),
            "seg_len":      float(seg["seg_len_frames"]),
            "n_active":     N,
        }


def collate(batch):
    B = len(batch)
    max_n = max(b["n_active"] for b in batch)
    D = batch[0]["inst_embs"].shape[1]

    inst_embs = torch.zeros(B, max_n, D)
    diff_ch   = torch.zeros(B, max_n, D)
    diff_vid  = torch.zeros(B, max_n, D)
    diff_stt  = torch.zeros(B, max_n, D)
    inst_scalars = torch.zeros(B, max_n, SCALAR_DIM)
    gt_bbox   = torch.zeros(B, max_n, 4)
    inst_mask = torch.zeros(B, max_n, dtype=torch.bool)

    seg_scalar = torch.zeros(B, SEG_SCALAR_DIM)
    channel_emb = torch.zeros(B, D)
    video_emb   = torch.zeros(B, D)
    stt_emb     = torch.zeros(B, D)
    channel_id  = torch.zeros(B, dtype=torch.long)
    seg_len     = torch.zeros(B)

    for bi, b in enumerate(batch):
        n = b["n_active"]
        inst_embs[bi, :n]    = b["inst_embs"]
        diff_ch[bi, :n]      = b["diff_ch"]
        diff_vid[bi, :n]     = b["diff_vid"]
        diff_stt[bi, :n]     = b["diff_stt"]
        inst_scalars[bi, :n] = b["inst_scalars"]
        gt_bbox[bi, :n]      = b["gt_bbox"]
        inst_mask[bi, :n]    = True
        seg_scalar[bi]   = b["seg_scalar"]
        channel_emb[bi]  = b["channel_emb"]
        video_emb[bi]    = b["video_emb"]
        stt_emb[bi]      = b["stt_emb"]
        channel_id[bi]   = b["channel_id"]
        seg_len[bi]      = b["seg_len"]

    return {
        "inst_embs":    inst_embs,
        "diff_ch":      diff_ch,
        "diff_vid":     diff_vid,
        "diff_stt":     diff_stt,
        "inst_scalars": inst_scalars,
        "gt_bbox":      gt_bbox,
        "inst_mask":    inst_mask,
        "seg_scalar":   seg_scalar,
        "channel_emb":  channel_emb,
        "video_emb":    video_emb,
        "stt_emb":      stt_emb,
        "channel_id":   channel_id,
        "seg_len":      seg_len,
    }


train_ds = SegmentDataset(train_entries, channel2id, text2emb, ch_stats)
eval_ds  = SegmentDataset(eval_entries,  channel2id, text2emb, ch_stats)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS_DL, collate_fn=collate,
                          pin_memory=True, drop_last=True)
eval_loader  = DataLoader(eval_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS_DL, collate_fn=collate,
                          pin_memory=True)
print(f"🚚 train: {len(train_loader):,}  eval: {len(eval_loader):,} batches")


segment 분해: 100%|██████████| 288/288 [00:00<00:00, 2796.46it/s]

📦 train: 307,294  eval: 15,451 segments
🚚 train: 600  eval: 31 batches


In [5]:
# %% 셀 3: COFS-style DETR refinement
# DETR-Refine 베이스 + 3-mode per-instance masking:
#   Mode A (Normal):   init_ref = inst_proj
#   Mode B (DN-DETR):  init_ref = GT + noise
#   Mode C (Context):  init_ref = clean GT, 모든 layer에서 GT 고정, loss 제외
D_MODEL = EMB_DIM
N_HEADS = 8
N_LAYERS = 6
FFN_DIM = D_MODEL * 4
DROPOUT = 0.1
N_CHANNELS = len(channels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class CofsLayoutModel(nn.Module):
    def __init__(self, emb_dim, n_channels,
                 d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                 ffn=FFN_DIM, dropout=DROPOUT):
        super().__init__()
        self.n_layers = n_layers
        self.channel_embed = nn.Embedding(n_channels, d_model)

        # Context projection
        self.ctx_ch_proj     = nn.Linear(emb_dim, d_model)
        self.ctx_vid_proj    = nn.Linear(emb_dim, d_model)
        self.ctx_stt_proj    = nn.Linear(emb_dim, d_model)
        self.ctx_scalar_proj = nn.Linear(SEG_SCALAR_DIM, d_model)

        # Instance projection
        self.diff_ch_proj     = nn.Linear(emb_dim, d_model)
        self.diff_vid_proj    = nn.Linear(emb_dim, d_model)
        self.diff_stt_proj    = nn.Linear(emb_dim, d_model)
        self.inst_scalar_proj = nn.Linear(SCALAR_DIM, d_model)

        self.type_embed = nn.Embedding(2, d_model)

        # 6 transformer layers (manual list for per-layer access)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=ffn,
                dropout=dropout, activation="gelu", batch_first=True,
                norm_first=True,
            ) for _ in range(n_layers)
        ])

        # Shared bbox heads (BLT factored)
        self.head_norm = nn.LayerNorm(d_model)
        self.cx_head = nn.Linear(d_model, GRID_W)
        self.cy_head = nn.Linear(d_model, GRID_H)
        self.w_head  = nn.Linear(d_model, GRID_W)
        self.h_head  = nn.Linear(d_model, GRID_H)
        for head in (self.cx_head, self.cy_head, self.w_head, self.h_head):
            nn.init.zeros_(head.bias)

        # Reference encoder: bbox (4-dim normalized) → D
        self.ref_encoder = nn.Sequential(
            nn.Linear(4, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        # zero-init: 학습 초기엔 ref_feat = 0 → baseline 거동과 동일하게 시작
        for layer in self.ref_encoder:
            if isinstance(layer, nn.Linear):
                nn.init.zeros_(layer.weight)
                nn.init.zeros_(layer.bias)

        # Initial reference projection: 인스턴스 feature → 초기 ref bbox
        self.init_ref_proj = nn.Linear(d_model, 4)
        nn.init.zeros_(self.init_ref_proj.weight)
        nn.init.zeros_(self.init_ref_proj.bias)

        # COFS-style 3-mode (training only, per-instance):
        #   co_prob: Mode C (clean GT context, loss 제외)
        #   dn_prob: Mode B (noisy GT, refinement 학습)
        #   나머지: Mode A (정상)
        self.co_prob = 0.2
        self.dn_prob = 0.3
        self.dn_noise_scale = 0.1

        # Context flag embedding: 모델이 어떤 인스턴스가 context(GT 주입)인지 알게 함
        # 0 = target, 1 = context. Mode C 인스턴스에만 context embedding 더해줌
        self.context_flag = nn.Embedding(2, d_model)
        nn.init.zeros_(self.context_flag.weight)  # 초기엔 영향 없게

        # softargmax indices for soft decode
        self.register_buffer("_x_arange", torch.arange(GRID_W).float())
        self.register_buffer("_y_arange", torch.arange(GRID_H).float())

    def _heads(self, inst_norm):
        cx_logits = self.cx_head(inst_norm)
        cy_logits = self.cy_head(inst_norm)
        w_logits  = self.w_head(inst_norm)
        h_logits  = self.h_head(inst_norm)
        return cx_logits, cy_logits, w_logits, h_logits

    def _soft_decode(self, cx_logits, cy_logits, w_logits, h_logits):
        prob_cx = F.softmax(cx_logits.float(), dim=-1)
        prob_cy = F.softmax(cy_logits.float(), dim=-1)
        prob_w  = F.softmax(w_logits.float(),  dim=-1)
        prob_h  = F.softmax(h_logits.float(),  dim=-1)
        cx = (prob_cx * self._x_arange).sum(-1) / float(GRID_W)
        cy = (prob_cy * self._y_arange).sum(-1) / float(GRID_H)
        w  = ((prob_w  * self._x_arange).sum(-1) + 1.0) / float(GRID_W)
        h  = ((prob_h  * self._y_arange).sum(-1) + 1.0) / float(GRID_H)
        return torch.stack([cx, cy, w, h], dim=-1)

    def _confidence(self, cx_logits, cy_logits, w_logits, h_logits):
        pk_cx = F.softmax(cx_logits.float(), -1).max(-1).values
        pk_cy = F.softmax(cy_logits.float(), -1).max(-1).values
        pk_w  = F.softmax(w_logits.float(),  -1).max(-1).values
        pk_h  = F.softmax(h_logits.float(),  -1).max(-1).values
        return pk_cx * pk_cy * pk_w * pk_h

    def _argmax_decode(self, cx_logits, cy_logits, w_logits, h_logits):
        cx = cx_logits.float().argmax(-1).float()
        cy = cy_logits.float().argmax(-1).float()
        w  = (w_logits.float().argmax(-1) + 1).float()
        h  = (h_logits.float().argmax(-1) + 1).float()
        return torch.stack([cx, cy, w, h], dim=-1)

    def forward(self, batch):
        B = batch["inst_embs"].size(0)
        N = batch["inst_embs"].size(1)

        inst_tok = (self.diff_ch_proj(batch["diff_ch"])
                    + self.diff_vid_proj(batch["diff_vid"])
                    + self.diff_stt_proj(batch["diff_stt"])
                    + self.inst_scalar_proj(batch["inst_scalars"]))
        inst_tok = inst_tok + self.type_embed.weight[1]

        ch_id_emb = self.channel_embed(batch["channel_id"])
        ctx_tok = (self.ctx_ch_proj(batch["channel_emb"])
                   + self.ctx_vid_proj(batch["video_emb"])
                   + self.ctx_stt_proj(batch["stt_emb"])
                   + self.ctx_scalar_proj(batch["seg_scalar"])
                   + ch_id_emb)
        ctx_tok = ctx_tok + self.type_embed.weight[0]
        ctx_tok = ctx_tok.unsqueeze(1)

        # 인스턴스 features에서 초기 ref 추정
        inst_ref = torch.sigmoid(self.init_ref_proj(inst_tok))

        # 3-mode per-instance assignment (학습 시만)
        co_mask = torch.zeros(B, N, dtype=torch.bool, device=inst_tok.device)
        dn_mask = torch.zeros(B, N, dtype=torch.bool, device=inst_tok.device)
        gt_norm = None
        if self.training and ("gt_bbox" in batch):
            gx0 = batch["gt_bbox"][..., 0].float()
            gy0 = batch["gt_bbox"][..., 1].float()
            gx1 = batch["gt_bbox"][..., 2].float()
            gy1 = batch["gt_bbox"][..., 3].float()
            gt_norm = torch.stack([
                ((gx0 + gx1) / 2.0) / float(GRID_W),
                ((gy0 + gy1) / 2.0) / float(GRID_H),
                ((gx1 - gx0).clamp(min=1.0)) / float(GRID_W),
                ((gy1 - gy0).clamp(min=1.0)) / float(GRID_H),
            ], dim=-1)
            rand = torch.rand(B, N, device=inst_tok.device)
            co_mask = (rand < self.co_prob) & batch["inst_mask"]
            dn_mask = ((rand >= self.co_prob) &
                       (rand < self.co_prob + self.dn_prob) &
                       batch["inst_mask"])
            noise = torch.randn_like(gt_norm) * self.dn_noise_scale
            noisy_ref = (gt_norm + noise).clamp(0.0, 1.0)

            init_ref = inst_ref
            init_ref = torch.where(dn_mask.unsqueeze(-1), noisy_ref, init_ref)
            init_ref = torch.where(co_mask.unsqueeze(-1), gt_norm,   init_ref)
            # context flag: Mode C 인스턴스에 flag embedding 더해서
            # 모델이 "이건 GT 주어진 인스턴스다"라고 알게 함
            inst_tok = inst_tok + self.context_flag(co_mask.long())
        else:
            init_ref = inst_ref

        ref_feat_init = self.ref_encoder(init_ref)
        inst_tok = inst_tok + ref_feat_init

        tokens = torch.cat([ctx_tok, inst_tok], dim=1)
        ctx_pad = torch.zeros(B, 1, dtype=torch.bool, device=tokens.device)
        pad_mask = torch.cat([ctx_pad, ~batch["inst_mask"]], dim=1)

        all_logits = []
        for k, layer in enumerate(self.layers):
            tokens = layer(tokens, src_key_padding_mask=pad_mask)
            inst_out = self.head_norm(tokens[:, 1:])

            cx_l, cy_l, w_l, h_l = self._heads(inst_out)
            all_logits.append((cx_l, cy_l, w_l, h_l))

            if k < self.n_layers - 1:
                ref = self._soft_decode(cx_l, cy_l, w_l, h_l)
                conf = self._confidence(cx_l, cy_l, w_l, h_l)
                gate = conf.unsqueeze(-1)
                # Mode C: 매 layer에서 ref를 GT로 고정, gate=1 (강한 신호)
                if self.training and gt_norm is not None:
                    ref = torch.where(co_mask.unsqueeze(-1), gt_norm, ref)
                    gate = torch.where(co_mask.unsqueeze(-1),
                                        torch.ones_like(gate), gate)
                ref_feat = self.ref_encoder(ref) * gate
                inst_part = tokens[:, 1:] + ref_feat
                tokens = torch.cat([tokens[:, :1], inst_part], dim=1)

        cx_f, cy_f, w_f, h_f = all_logits[-1]
        params = self._argmax_decode(cx_f, cy_f, w_f, h_f)
        return all_logits, params, co_mask, dn_mask


model = CofsLayoutModel(EMB_DIM, N_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"🧠 모델 params: {n_params/1e6:.2f}M | D={D_MODEL} L={N_LAYERS} H={N_HEADS}")
print(f"   3-mode: co_prob={model.co_prob} dn_prob={model.dn_prob} normal={1-model.co_prob-model.dn_prob:.2f}")
print(f"   Mode C: GT를 clean ref로 주입, 매 layer GT 고정, loss 제외")
print(f"   Mode B: GT+noise(σ={model.dn_noise_scale})를 ref로 주입, loss 포함")
print(f"   Mode A: inst_proj 예측을 ref로, loss 포함")


🧠 모델 params: 83.36M | D=1024 L=6 H=8
   3-mode: co_prob=0.2 dn_prob=0.3 normal=0.50
   Mode C: GT를 clean ref로 주입, 매 layer GT 고정, loss 제외
   Mode B: GT+noise(σ=0.1)를 ref로 주입, loss 포함
   Mode A: inst_proj 예측을 ref로, loss 포함


In [ ]:
# %% 셀 4: 학습 (deep supervision, Mode C 인스턴스는 loss 제외)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 0.01
CX_LOSS_WEIGHT = 1.0
CY_LOSS_WEIGHT = 1.0
W_LOSS_WEIGHT  = 1.0
H_LOSS_WEIGHT  = 1.0
LABEL_SMOOTH_SIGMA = 1.0
LAYER_WEIGHT_RATIO = 1.0
SAVE_DIR = "./model/8_layout_cofs"
os.makedirs(SAVE_DIR, exist_ok=True)

_x_idx_f = torch.arange(GRID_W, device=device, dtype=torch.float32)
_y_idx_f = torch.arange(GRID_H, device=device, dtype=torch.float32)


def _gaussian_ce(logits, gt_target_idx_f, idx_f, sigma):
    g = idx_f.view(1, 1, -1)
    gauss = torch.exp(-0.5 * ((g - gt_target_idx_f.unsqueeze(-1)) / sigma) ** 2)
    target = gauss / gauss.sum(-1, keepdim=True).clamp(min=1e-8)
    log_p = F.log_softmax(logits.float(), dim=-1)
    return -(target * log_p).sum(-1)


def per_layer_losses(cx_l, cy_l, w_l, h_l, gt_bbox, sigma=LABEL_SMOOTH_SIGMA):
    gx0 = gt_bbox[..., 0].float(); gy0 = gt_bbox[..., 1].float()
    gx1 = gt_bbox[..., 2].float(); gy1 = gt_bbox[..., 3].float()
    gt_cx = (gx0 + gx1) / 2.0
    gt_cy = (gy0 + gy1) / 2.0
    gt_w  = (gx1 - gx0).clamp(min=1.0) - 1.0
    gt_h  = (gy1 - gy0).clamp(min=1.0) - 1.0
    return (
        _gaussian_ce(cx_l, gt_cx, _x_idx_f, sigma),
        _gaussian_ce(cy_l, gt_cy, _y_idx_f, sigma),
        _gaussian_ce(w_l,  gt_w,  _x_idx_f, sigma),
        _gaussian_ce(h_l,  gt_h,  _y_idx_f, sigma),
    )


def aggregate_per_token(loss_tok, loss_mask, seg_len_w):
    if not loss_mask.any():
        return loss_tok.sum() * 0.0
    m = loss_mask.float()
    counts = m.sum(dim=1).clamp(min=1.0)
    seg_loss = (loss_tok * m).sum(dim=1) / counts
    # segment의 target 수가 0인 경우 weight 0 처리
    has_target = (m.sum(dim=1) > 0).float()
    w_n = seg_len_w * has_target
    w_n = w_n / w_n.sum().clamp(min=1e-8)
    return (seg_loss * w_n).sum()


def compute_losses_deep(all_logits, gt_bbox, loss_mask, seg_len_w,
                        layer_weight_ratio=LAYER_WEIGHT_RATIO):
    L = len(all_logits)
    weights = [layer_weight_ratio ** (L - 1 - k) for k in range(L)]
    s = sum(weights)
    weights = [w / s for w in weights]

    total = {"cx": 0.0, "cy": 0.0, "w": 0.0, "h": 0.0}
    for k, (cx_l, cy_l, w_l, h_l) in enumerate(all_logits):
        l_cx, l_cy, l_w, l_h = per_layer_losses(cx_l, cy_l, w_l, h_l, gt_bbox)
        wk = weights[k]
        total["cx"] = total["cx"] + wk * aggregate_per_token(l_cx, loss_mask, seg_len_w)
        total["cy"] = total["cy"] + wk * aggregate_per_token(l_cy, loss_mask, seg_len_w)
        total["w"]  = total["w"]  + wk * aggregate_per_token(l_w,  loss_mask, seg_len_w)
        total["h"]  = total["h"]  + wk * aggregate_per_token(l_h,  loss_mask, seg_len_w)
    return total["cx"], total["cy"], total["w"], total["h"]


@torch.no_grad()
def compute_metrics(pred_params, gt_bbox, inst_mask, seg_len_w,
                    cx_logits=None, cy_logits=None):
    if not inst_mask.any():
        return None
    pred = pred_params.float()
    gt   = gt_bbox.float()
    cx, cy, w, h = pred.unbind(-1)
    gx0, gy0, gx1, gy1 = gt.unbind(-1)
    px0 = cx - w/2; py0 = cy - h/2
    px1 = cx + w/2; py1 = cy + h/2
    ix0 = torch.max(px0, gx0); iy0 = torch.max(py0, gy0)
    ix1 = torch.min(px1, gx1); iy1 = torch.min(py1, gy1)
    inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
    pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
    ga = (gx1 - gx0).clamp(min=0) * (gy1 - gy0).clamp(min=0)
    iou = inter / (pa + ga - inter + 1e-8)
    gt_cx = (gx0 + gx1) / 2; gt_cy = (gy0 + gy1) / 2
    gt_w  = (gx1 - gx0).clamp(min=1.0); gt_h = (gy1 - gy0).clamp(min=1.0)

    m = inst_mask.float()
    counts = m.sum(dim=1).clamp(min=1.0)
    w_n = seg_len_w / seg_len_w.sum().clamp(min=1e-8)
    miou       = ((iou * m).sum(1) / counts * w_n).sum().item()
    iou_50     = (((iou > 0.5).float() * m).sum(1) / counts * w_n).sum().item()
    iou_75     = (((iou > 0.75).float() * m).sum(1) / counts * w_n).sum().item()
    err_cx     = (((cx - gt_cx).abs() * m).sum(1) / counts * w_n).sum().item()
    err_cy     = (((cy - gt_cy).abs() * m).sum(1) / counts * w_n).sum().item()
    err_w      = (((w  - gt_w).abs()  * m).sum(1) / counts * w_n).sum().item()
    err_h      = (((h  - gt_h).abs()  * m).sum(1) / counts * w_n).sum().item()
    out = {
        "mIoU": miou, "IoU@0.5": iou_50, "IoU@0.75": iou_75,
        "errCX": err_cx, "errCY": err_cy, "errW": err_w, "errH": err_h,
    }
    if cx_logits is not None and cy_logits is not None:
        prob_cx = F.softmax(cx_logits.float(), dim=-1)
        prob_cy = F.softmax(cy_logits.float(), dim=-1)
        peak_cx = prob_cx.max(-1).values
        peak_cy = prob_cy.max(-1).values
        gt_cx_idx = ((gx0 + gx1) / 2.0).clamp(0, GRID_W - 1).long()
        gt_cy_idx = ((gy0 + gy1) / 2.0).clamp(0, GRID_H - 1).long()
        top1_cx = (prob_cx.argmax(-1) == gt_cx_idx).float()
        top1_cy = (prob_cy.argmax(-1) == gt_cy_idx).float()
        out.update({
            "peak_x":  ((peak_cx * m).sum(1) / counts * w_n).sum().item(),
            "peak_y":  ((peak_cy * m).sum(1) / counts * w_n).sum().item(),
            "cx@1":    ((top1_cx * m).sum(1) / counts * w_n).sum().item(),
            "cy@1":    ((top1_cy * m).sum(1) / counts * w_n).sum().item(),
        })
    return out


@torch.no_grad()
def metrics_with_buckets(all_logits, gt_bbox, inst_mask, seg_len_w):
    layer_results = []
    for cx_l, cy_l, w_l, h_l in all_logits:
        cx = cx_l.float().argmax(-1).float()
        cy = cy_l.float().argmax(-1).float()
        w  = (w_l.float().argmax(-1) + 1).float()
        h  = (h_l.float().argmax(-1) + 1).float()
        params = torch.stack([cx, cy, w, h], dim=-1)
        m_dict = compute_metrics(params, gt_bbox, inst_mask, seg_len_w,
                                  cx_logits=cx_l, cy_logits=cy_l)
        if m_dict is None:
            layer_results.append(None); continue

        prob_cx = F.softmax(cx_l.float(), dim=-1)
        prob_cy = F.softmax(cy_l.float(), dim=-1)
        peak_cx = prob_cx.max(-1).values
        peak_cy = prob_cy.max(-1).values
        conf = peak_cx * peak_cy

        gx0, gy0, gx1, gy1 = gt_bbox.float().unbind(-1)
        px0 = cx - w/2; py0 = cy - h/2
        px1 = cx + w/2; py1 = cy + h/2
        ix0 = torch.max(px0, gx0); iy0 = torch.max(py0, gy0)
        ix1 = torch.min(px1, gx1); iy1 = torch.min(py1, gy1)
        inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
        pa = (px1 - px0).clamp(min=0) * (py1 - py0).clamp(min=0)
        ga = (gx1 - gx0).clamp(min=0) * (gy1 - gy0).clamp(min=0)
        iou = inter / (pa + ga - inter + 1e-8)

        m = inst_mask.float()
        seg_w_exp = seg_len_w.unsqueeze(-1).expand_as(conf)
        m_total = (m * seg_w_exp).sum().clamp(min=1e-8)
        for kk in range(10):
            lo, hi = kk * 0.1, (kk + 1) * 0.1
            if kk < 9:
                b_mask = (conf >= lo) & (conf < hi)
            else:
                b_mask = (conf >= lo)
            bm = (b_mask & inst_mask).float() * seg_w_exp
            bm_sum = bm.sum().clamp(min=1e-8)
            mi   = (iou * bm).sum() / bm_sum
            frac = bm.sum() / m_total
            m_dict[f"miou[{lo:.1f}]"] = mi.item()
            m_dict[f"frac[{lo:.1f}]"] = frac.item()
        layer_results.append(m_dict)
    return layer_results


def batch_to_device(batch, device):
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_score = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tot = {"loss":0.0, "cx":0.0, "cy":0.0, "w":0.0, "h":0.0}
    mode_counts = {"co": 0, "dn": 0, "normal": 0, "active": 0}
    n_b = 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        batch = batch_to_device(batch, device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            all_logits, params, co_mask, dn_mask = model(batch)
            # Mode C 인스턴스는 loss에서 제외
            loss_mask = batch["inst_mask"] & (~co_mask)
            loss_cx, loss_cy, loss_w, loss_h = compute_losses_deep(
                all_logits, batch["gt_bbox"], loss_mask, batch["seg_len"],
            )
            loss = (CX_LOSS_WEIGHT * loss_cx +
                    CY_LOSS_WEIGHT * loss_cy +
                    W_LOSS_WEIGHT  * loss_w  +
                    H_LOSS_WEIGHT  * loss_h)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot["loss"] += loss.item()
        tot["cx"]   += loss_cx.item();  tot["cy"] += loss_cy.item()
        tot["w"]    += loss_w.item();   tot["h"]  += loss_h.item()
        with torch.no_grad():
            active = batch["inst_mask"]
            mode_counts["active"] += active.sum().item()
            mode_counts["co"]     += co_mask.sum().item()
            mode_counts["dn"]     += dn_mask.sum().item()
            mode_counts["normal"] += (active & ~co_mask & ~dn_mask).sum().item()
        n_b += 1

    model.eval()
    eval_loss_sum = 0.0
    n_eb = 0
    all_layer_metrics = [[] for _ in range(N_LAYERS)]
    with torch.no_grad():
        for batch in eval_loader:
            batch = batch_to_device(batch, device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                all_logits, params, co_mask, dn_mask = model(batch)
                # eval에서는 co_mask 전부 0 → loss_mask == inst_mask
                loss_cx, loss_cy, loss_w, loss_h = compute_losses_deep(
                    all_logits, batch["gt_bbox"], batch["inst_mask"], batch["seg_len"],
                )
                eloss = (CX_LOSS_WEIGHT * loss_cx + CY_LOSS_WEIGHT * loss_cy +
                         W_LOSS_WEIGHT  * loss_w  + H_LOSS_WEIGHT  * loss_h)
            layer_metrics = metrics_with_buckets(all_logits, batch["gt_bbox"],
                                                  batch["inst_mask"], batch["seg_len"])
            for k, lm in enumerate(layer_metrics):
                if lm is not None:
                    all_layer_metrics[k].append((lm, batch["seg_len"].sum().item()))
            eval_loss_sum += eloss.item()
            n_eb += 1

    def agg_dicts(metric_list):
        tw = sum(w for _, w in metric_list) or 1.0
        keys = list(metric_list[0][0].keys())
        return {k: sum(d[k] * w for d, w in metric_list) / tw for k in keys}

    layer_aggs = [agg_dicts(lst) for lst in all_layer_metrics]
    final_agg  = layer_aggs[-1]

    scheduler.step()
    bands = [f"{kk*0.1:.1f}" for kk in range(10)]

    act = max(mode_counts["active"], 1)
    co_pct  = mode_counts["co"] / act * 100
    dn_pct  = mode_counts["dn"] / act * 100
    nor_pct = mode_counts["normal"] / act * 100

    msg = ("[{ep}/{ne}]  train={tl:.4f} (cx={cx_l:.3f} cy={cy_l:.3f} w={wl:.3f} h={hl:.3f})  "
           "eval={el:.4f}  mIoU={miou:.4f}  cx@1={c_x:.3f} cy@1={c_y:.3f}  "
           "IoU@0.5={i50:.3f}  errCX={ecx:.2f} errCY={ecy:.2f} errW={ew:.2f} errH={eh:.2f}  "
           "pkX={px:.3f} pkY={py:.3f}  lr={lr:.2e}"
           ).format(ep=epoch, ne=EPOCHS,
                    tl=tot["loss"]/max(n_b,1),
                    cx_l=tot["cx"]/max(n_b,1), cy_l=tot["cy"]/max(n_b,1),
                    wl=tot["w"]/max(n_b,1),    hl=tot["h"]/max(n_b,1),
                    el=eval_loss_sum/max(n_eb,1),
                    miou=final_agg["mIoU"],
                    c_x=final_agg.get("cx@1", 0.0), c_y=final_agg.get("cy@1", 0.0),
                    i50=final_agg["IoU@0.5"],
                    ecx=final_agg["errCX"], ecy=final_agg["errCY"],
                    ew=final_agg["errW"], eh=final_agg["errH"],
                    px=final_agg.get("peak_x", 0.0), py=final_agg.get("peak_y", 0.0),
                    lr=scheduler.get_last_lr()[0])
    print(msg)
    print(f"           modes: co={co_pct:.1f}%  dn={dn_pct:.1f}%  normal={nor_pct:.1f}%")

    layer_miou_str = " → ".join(f"L{k+1}:{a['mIoU']:.3f}" for k, a in enumerate(layer_aggs))
    print(f"           layers: {layer_miou_str}")

    for k, a in enumerate(layer_aggs):
        bucket_str = "  ".join(
            f"{b}:{a.get(f'miou[{b}]', 0.0):.2f}[{a.get(f'frac[{b}]', 0.0)*100:.0f}%]"
            for b in bands
        )
        print(f"           L{k+1}: {bucket_str}")
    agg = final_agg

    if agg["mIoU"] > best_score:
        best_score = agg["mIoU"]
        torch.save({"model": model.state_dict(), "epoch": epoch,
                    "mIoU": best_score},
                   os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (mIoU={best_score:.4f})")


[1/50]  train=12.1191 (cx=2.733 cy=3.480 w=4.016 h=1.890)  eval=11.8272  mIoU=0.1453  cx@1=0.504 cy@1=0.151  IoU@0.5=0.128  errCX=6.04 errCY=15.54 errW=15.87 errH=1.13  pkX=0.243 pkY=0.117  lr=9.99e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.141 → L2:0.142 → L3:0.140 → L4:0.146 → L5:0.148 → L6:0.145
           L1: 0.0:0.11[99%]  0.1:0.09[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.11[99%]  0.1:0.04[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.11[99%]  0.1:0.05[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.12[99%]  0.1:0.10[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.12[99%]  0.1:0.10[1%]  

[2/50]  train=11.5791 (cx=2.605 cy=3.267 w=3.876 h=1.831)  eval=11.6598  mIoU=0.1577  cx@1=0.526 cy@1=0.171  IoU@0.5=0.150  errCX=6.20 errCY=14.52 errW=15.05 errH=1.19  pkX=0.230 pkY=0.137  lr=9.96e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.137 → L2:0.149 → L3:0.154 → L4:0.157 → L5:0.158 → L6:0.158
           L1: 0.0:0.10[99%]  0.1:0.11[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.11[99%]  0.1:0.11[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.12[99%]  0.1:0.10[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.12[99%]  0.1:0.12[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.12[99%]  0.1:0.12[1%]  

[3/50]  train=11.1775 (cx=2.519 cy=3.149 w=3.710 h=1.799)  eval=11.1615  mIoU=0.2113  cx@1=0.551 cy@1=0.200  IoU@0.5=0.218  errCX=5.37 errCY=11.61 errW=11.36 errH=0.99  pkX=0.246 pkY=0.148  lr=9.91e-05
           modes: co=20.0%  dn=30.0%  normal=50.1%
           layers: L1:0.185 → L2:0.203 → L3:0.209 → L4:0.212 → L5:0.210 → L6:0.211
           L1: 0.0:0.14[99%]  0.1:0.11[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.16[98%]  0.1:0.21[2%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.17[97%]  0.1:0.34[3%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.17[97%]  0.1:0.34[3%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.17[97%]  0.1:0.33[3%]  

[4/50]  train=10.9080 (cx=2.465 cy=3.077 w=3.591 h=1.776)  eval=11.0558  mIoU=0.2246  cx@1=0.557 cy@1=0.198  IoU@0.5=0.235  errCX=4.91 errCY=11.46 errW=10.65 errH=0.96  pkX=0.275 pkY=0.155  lr=9.84e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.196 → L2:0.216 → L3:0.224 → L4:0.226 → L5:0.224 → L6:0.225
           L1: 0.0:0.16[99%]  0.1:0.10[1%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.18[97%]  0.1:0.29[3%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.18[96%]  0.1:0.26[4%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.19[96%]  0.1:0.24[4%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.18[96%]  0.1:0.22[4%]  

[5/50]  train=10.7467 (cx=2.436 cy=3.026 w=3.523 h=1.761)  eval=10.9397  mIoU=0.2360  cx@1=0.570 cy@1=0.240  IoU@0.5=0.239  errCX=4.90 errCY=11.15 errW=10.00 errH=0.93  pkX=0.276 pkY=0.169  lr=9.76e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.205 → L2:0.237 → L3:0.236 → L4:0.237 → L5:0.234 → L6:0.236
           L1: 0.0:0.16[98%]  0.1:0.18[2%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.19[96%]  0.1:0.32[4%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.18[95%]  0.1:0.35[5%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.18[94%]  0.1:0.35[6%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.18[94%]  0.1:0.37[6%]  

...

[19/50]  train=9.5766 (cx=2.192 cy=2.650 w=3.071 h=1.663)  eval=10.6772  mIoU=0.2920  cx@1=0.565 cy@1=0.260  IoU@0.5=0.303  errCX=4.79 errCY=8.95 errW=9.09 errH=0.82  pkX=0.287 pkY=0.210  lr=6.84e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.251 → L2:0.281 → L3:0.287 → L4:0.294 → L5:0.290 → L6:0.292
           L1: 0.0:0.20[95%]  0.1:0.41[5%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.21[89%]  0.1:0.57[11%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.20[85%]  0.1:0.61[15%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.21[84%]  0.1:0.54[16%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.20[84%]  0.1:0.53[16%]

[20/50]  train=9.5329 (cx=2.186 cy=2.634 w=3.054 h=1.658)  eval=10.7294  mIoU=0.2793  cx@1=0.563 cy@1=0.250  IoU@0.5=0.291  errCX=4.65 errCY=8.92 errW=8.83 errH=0.83  pkX=0.294 pkY=0.213  lr=6.55e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.251 → L2:0.284 → L3:0.280 → L4:0.281 → L5:0.281 → L6:0.279
           L1: 0.0:0.20[95%]  0.1:0.43[5%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.20[88%]  0.1:0.55[12%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.19[83%]  0.1:0.52[17%]  0.2:0.09[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.19[82%]  0.1:0.50[18%]  0.2:0.06[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.19[82%]  0.1:0.49[18%]

[21/50]  train=9.4715 (cx=2.172 cy=2.616 w=3.031 h=1.653)  eval=10.7873  mIoU=0.2865  cx@1=0.560 cy@1=0.253  IoU@0.5=0.303  errCX=4.98 errCY=8.69 errW=8.98 errH=0.82  pkX=0.284 pkY=0.215  lr=6.24e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.249 → L2:0.276 → L3:0.280 → L4:0.281 → L5:0.284 → L6:0.286
           L1: 0.0:0.19[95%]  0.1:0.37[5%]  0.2:0.06[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.20[90%]  0.1:0.49[10%]  0.2:0.06[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.20[85%]  0.1:0.51[15%]  0.2:0.06[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.20[83%]  0.1:0.49[16%]  0.2:0.06[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.20[84%]  0.1:0.48[16%]

[22/50]  train=9.3999 (cx=2.161 cy=2.590 w=3.000 h=1.649)  eval=10.8103  mIoU=0.2861  cx@1=0.549 cy@1=0.247  IoU@0.5=0.303  errCX=4.79 errCY=9.45 errW=9.26 errH=0.86  pkX=0.296 pkY=0.219  lr=5.94e-05
           modes: co=20.0%  dn=30.0%  normal=50.0%
           layers: L1:0.245 → L2:0.276 → L3:0.283 → L4:0.286 → L5:0.284 → L6:0.286
           L1: 0.0:0.19[94%]  0.1:0.33[6%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L2: 0.0:0.20[87%]  0.1:0.52[13%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L3: 0.0:0.20[83%]  0.1:0.50[17%]  0.2:0.02[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L4: 0.0:0.21[81%]  0.1:0.46[19%]  0.2:0.00[0%]  0.3:0.00[0%]  0.4:0.00[0%]  0.5:0.00[0%]  0.6:0.00[0%]  0.7:0.00[0%]  0.8:0.00[0%]  0.9:0.00[0%]
           L5: 0.0:0.20[81%]  0.1:0.47[19%]

[23/50] train:  31%|███       | 184/600 [00:22<00:44,  9.30it/s]